# Chronological Data Splitting

**Goal:** create reproducible train, validation, and test datasets for delivery-delay prediction.

- Split by `order_purchase_timestamp` (70% / 15% / 15%), not randomly.
- Keep orders sharing the same timestamp in the same split.
- Exclude targets and metadata from predictor matrices.
- Save complete split datasets as Parquet, plus a JSON manifest.

## 1. Imports and paths

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "03_feature_engineering.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "splits"

assert INPUT_PATH.is_file(), f"Missing input: {INPUT_PATH}. Run notebook 03 first."
dataset = pd.read_parquet(INPUT_PATH, engine="pyarrow")
print(f"Loaded {len(dataset):,} rows and {len(dataset.columns)} columns")
display(dataset.head(3))

Loaded 96,470 rows and 19 columns


,order_id,delivery_time_days,is_delayed,purchase_month,purchase_weekday,purchase_hour,purchase_year,purchase_is_weekend,items_quantity,unique_products,sellers_quantity,total_price,total_freight,min_distance_km,avg_distance_km,max_distance_km,all_sellers_same_state,has_complete_coordinates,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,0,10,0,10,2017,0,1,1,1,29.99,8.72,18.681711,18.681711,18.681711,True,True,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,0,7,1,20,2018,0,1,1,1,118.70,22.76,861.035367,861.035367,861.035367,False,True,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,0,8,2,8,2018,0,1,1,1,159.90,19.22,514.547140,514.547140,514.547140,False,True,2018-08-08 08:38:49


## 2. Explicit feature contract

This allowlist is intentionally repeated from notebook 03. Any schema change requires review before training.

In [2]:
FEATURE_COLUMNS = [
    "purchase_month", "purchase_weekday", "purchase_hour", "purchase_year",
    "purchase_is_weekend", "items_quantity", "unique_products", "sellers_quantity",
    "total_price", "total_freight",
    "min_distance_km", "avg_distance_km", "max_distance_km",
    "has_complete_coordinates", "all_sellers_same_state",
]
TARGET_COLUMNS = ["delivery_time_days", "is_delayed"]
METADATA_COLUMNS = ["order_id", "order_purchase_timestamp"]
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS + METADATA_COLUMNS

assert len(EXPECTED_COLUMNS) == len(set(EXPECTED_COLUMNS)), "Overlapping column groups"
assert set(dataset.columns) == set(EXPECTED_COLUMNS), (
    f"Missing: {set(EXPECTED_COLUMNS) - set(dataset.columns)}; "
    f"unexpected: {set(dataset.columns) - set(EXPECTED_COLUMNS)}"
)
assert dataset["order_id"].is_unique
assert dataset["order_purchase_timestamp"].notna().all()
assert dataset[TARGET_COLUMNS].notna().all().all()
assert dataset["is_delayed"].isin([0, 1]).all()
assert len(dataset) >= 3

dataset["order_purchase_timestamp"] = pd.to_datetime(
    dataset["order_purchase_timestamp"], errors="raise"
)
print("Feature contract and basic data integrity checks passed")

Feature contract and basic data integrity checks passed


## 3. Chronological split

Sort by purchase timestamp, then `order_id` for deterministic ordering. The nominal cutoffs are at 70% and 85% of rows. Each cutoff is moved to the **right end of its timestamp group**, so no purchase timestamp appears in multiple splits. Consequently, the realized ratios may differ slightly from 70/15/15.

In [3]:
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15

ordered = dataset.sort_values(
    ["order_purchase_timestamp", "order_id"], kind="stable"
).reset_index(drop=True)

timestamps = ordered["order_purchase_timestamp"]
group_ends = np.flatnonzero(timestamps.ne(timestamps.shift(-1)).to_numpy()) + 1
n_rows = len(ordered)

train_cut = int(group_ends[np.searchsorted(group_ends, int(np.ceil(n_rows * TRAIN_RATIO)))])
validation_cut = int(group_ends[np.searchsorted(
    group_ends, int(np.ceil(n_rows * (TRAIN_RATIO + VALIDATION_RATIO)))
)])

assert 0 < train_cut < validation_cut < n_rows, (
    "Cannot form three nonempty timestamp-disjoint partitions. "
    "Inspect timestamp cardinality or revise split ratios."
)

splits = {
    "train": ordered.iloc[:train_cut].copy().reset_index(drop=True),
    "validation": ordered.iloc[train_cut:validation_cut].copy().reset_index(drop=True),
    "test": ordered.iloc[validation_cut:].copy().reset_index(drop=True),
}

for name, frame in splits.items():
    print(f"{name:10} {len(frame):>7,} ({len(frame)/n_rows:.2%}) "
          f"| {frame['order_purchase_timestamp'].min()} → "
          f"{frame['order_purchase_timestamp'].max()}")

train       67,529 (70.00%) | 2016-09-15 12:16:38 → 2018-04-15 20:12:35
validation  14,471 (15.00%) | 2018-04-15 20:17:11 → 2018-06-21 08:41:07
test        14,470 (15.00%) | 2018-06-21 08:45:30 → 2018-08-29 15:00:37


## 4. Leakage and split integrity checks

These checks prevent row overlap, reversed time windows, and target/metadata leakage into the feature matrix. They do **not** prove that all upstream features were known at purchase time.

In [4]:
assert sum(len(frame) for frame in splits.values()) == len(dataset)

ids = {name: set(frame["order_id"]) for name, frame in splits.items()}
assert ids["train"].isdisjoint(ids["validation"])
assert ids["train"].isdisjoint(ids["test"])
assert ids["validation"].isdisjoint(ids["test"])
assert set.union(*ids.values()) == set(dataset["order_id"])

assert (splits["train"]["order_purchase_timestamp"].max()
        < splits["validation"]["order_purchase_timestamp"].min())
assert (splits["validation"]["order_purchase_timestamp"].max()
        < splits["test"]["order_purchase_timestamp"].min())

for name, frame in splits.items():
    assert frame["order_purchase_timestamp"].is_monotonic_increasing
    assert frame["order_id"].is_unique
    assert set(frame.columns) == set(EXPECTED_COLUMNS)

X = {name: frame[FEATURE_COLUMNS].copy() for name, frame in splits.items()}
y = {name: frame[TARGET_COLUMNS].copy() for name, frame in splits.items()}

for name in splits:
    assert not set(X[name].columns).intersection(TARGET_COLUMNS + METADATA_COLUMNS)
    assert len(X[name]) == len(y[name]) == len(splits[name])
    assert X[name].index.equals(y[name].index)

print("All split, chronological-order and schema-leakage checks passed")

All split, chronological-order and schema-leakage checks passed


## 5. Target distribution and missing-value diagnostics

Compare target prevalence and missingness across periods. Chronological splitting intentionally does **not** stratify on `is_delayed`: stratification could destroy the temporal structure. Differences across splits may reflect genuine temporal drift.

In [5]:
summary = pd.DataFrame([
    {
        "split": name,
        "orders": len(frame),
        "share": len(frame) / n_rows,
        "first_purchase": frame["order_purchase_timestamp"].min(),
        "last_purchase": frame["order_purchase_timestamp"].max(),
        "delay_rate": frame["is_delayed"].mean(),
        "mean_delivery_days": frame["delivery_time_days"].mean(),
        "rows_with_missing_features": frame[FEATURE_COLUMNS].isna().any(axis=1).mean(),
    }
    for name, frame in splits.items()
]).set_index("split")
display(summary)

missing_by_split = pd.DataFrame({
    name: frame[FEATURE_COLUMNS].isna().mean().mul(100)
    for name, frame in splits.items()
})
display(missing_by_split.loc[missing_by_split.gt(0).any(axis=1)].round(2))

print("Note: check for target drift and sparse positives before choosing metrics.")

,orders,share,first_purchase,last_purchase,delay_rate,mean_delivery_days,rows_with_missing_features
split,,,,,,,
train,67529,0.700000,2016-09-15 12:16:38,2018-04-15 20:12:35,0.078337,13.824217,0.005124
validation,14471,0.150005,2018-04-15 20:17:11,2018-06-21 08:41:07,0.043121,10.840081,0.005114
test,14470,0.149995,2018-06-21 08:45:30,2018-08-29 15:00:37,0.042847,8.368267,0.004216


,train,validation,test
min_distance_km,0.51,0.51,0.42
avg_distance_km,0.51,0.51,0.42
max_distance_km,0.51,0.51,0.42


Note: check for target drift and sparse positives before choosing metrics.


In [8]:
for name, frame in splits.items():
    counts = frame["is_delayed"].value_counts()
    assert counts.get(0, 0) > 0
    assert counts.get(1, 0) > 0
    print(name, counts.to_dict())

train {0: 62239, 1: 5290}
validation {0: 13847, 1: 624}
test {0: 13850, 1: 620}


## 6. Persist and verify

Each Parquet contains the **full** split (features, targets, and metadata) so subsequent notebooks can reconstruct `X` and `y` using the allowlist. The manifest records boundaries, sizes, and the feature contract. Do not use the test split for hyperparameter tuning.

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, frame in splits.items():
    path = OUTPUT_DIR / f"{name}.parquet"
    frame.to_parquet(path, index=False, engine="pyarrow")
    restored = pd.read_parquet(path, engine="pyarrow")
    pd.testing.assert_frame_equal(frame, restored, check_dtype=True)
    print(f"Verified {path}")

manifest = {
    "source": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "split_strategy": "chronological; no shared purchase timestamps across splits",
    "nominal_ratios": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "feature_columns": FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "metadata_columns": METADATA_COLUMNS,
    "splits": {
        name: {
            "path": str((OUTPUT_DIR / f"{name}.parquet").relative_to(PROJECT_ROOT)),
            "rows": len(frame),
            "first_purchase": frame["order_purchase_timestamp"].min().isoformat(),
            "last_purchase": frame["order_purchase_timestamp"].max().isoformat(),
            "delay_rate": float(frame["is_delayed"].mean()),
        }
        for name, frame in splits.items()
    },
}
manifest_path = OUTPUT_DIR / "split_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
assert json.loads(manifest_path.read_text(encoding="utf-8")) == manifest
print(f"Verified {manifest_path}")

Verified /home/arthur/Desktop/delivery-intelligence/data/processed/splits/train.parquet
Verified /home/arthur/Desktop/delivery-intelligence/data/processed/splits/validation.parquet
Verified /home/arthur/Desktop/delivery-intelligence/data/processed/splits/test.parquet
Verified /home/arthur/Desktop/delivery-intelligence/data/processed/splits/split_manifest.json


## 7. Findings and conclusions

After running all cells, interpret the printed summary above:

1. **Chronological validity:** all training orders precede validation orders, which precede test orders; timestamp ties never cross boundaries.
2. **Leakage controls:** target and metadata columns are excluded from predictor matrices. No preprocessing statistics are learned from validation or test data.
3. **Temporal drift:** compare `delay_rate`, mean delivery time, and missingness across the three periods; differences can change generalization performance.
4. **Population limitation:** notebook 03 uses delivered orders with observed delivery outcomes. Evaluation here therefore applies to this selected population, not necessarily all orders placed.
5. **Outcome availability caveat:** the split uses **purchase time**, but training labels become available only after delivery. A realistic historical deployment simulation would also enforce a label-availability cutoff or temporal gap before each evaluation window.
6. **Next notebook:** fit preprocessing and a baseline classifier using `train` only; select hyperparameters with `validation`; reserve `test` for final evaluation.

**Status:** notebook code prepared; findings that depend on actual split dates and target rates must be confirmed by executing the notebook with the processed Parquet file.